In [1]:
# ================================================================
# BYM2 (FULL COVARIATE, phi_k FIXED = 0.7)
# Conjugate PG + Gaussian MCMC
# SAVE theta + tau
# ================================================================

import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm import tqdm
from pathlib import Path
import os

from scipy.spatial.distance import pdist, squareform
from scipy.sparse import csr_matrix, coo_matrix, diags, bmat
from sksparse.cholmod import cholesky
from polyagamma import random_polyagamma
import pyreadr

# ================================================================
# Paths
# ================================================================
os.chdir(Path.cwd().parent)

# ================================================================
# Load data (remove isolated points)
# ================================================================
no_nbs = np.array([
    57,170,236,269,343,685,946,947,989,
    1037,1084,1090,1109,1118,1127,1176,1203
]) - 1

snow_cleaned_full = pyreadr.read_r("snow_cleaned_full.Rda")
snow_cleaned_full = list(snow_cleaned_full.values())[0]

all_y = snow_cleaned_full.drop(index=no_nbs).reset_index(drop=True)

coords = all_y.iloc[:, :2].to_numpy()
y      = all_y.iloc[:, 2:].to_numpy()

S, TT = y.shape
period = 52

print(f"[INFO] Spatial locations used: {S}")

# ================================================================
# Global time trend (scale FIRST)
# ================================================================
t_full = np.arange(1, TT + 1)
t_trend_full = (t_full - t_full.mean()) / t_full.std(ddof=0)

# ================================================================
# Build adjacency
# ================================================================
gdf = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(coords[:, 0], coords[:, 1]),
    crs="EPSG:4326"
)

gdf_aeqd = gdf.to_crs("+proj=aeqd +lat_0=90 +lon_0=-100")
xy = np.vstack([gdf_aeqd.geometry.x, gdf_aeqd.geometry.y]).T

dif = xy[1] - xy[2]
theta_rot = np.arctan2(dif[1], dif[0])
R = np.array([
    [np.cos(theta_rot), -np.sin(theta_rot)],
    [np.sin(theta_rot),  np.cos(theta_rot)]
])

rotated = (xy @ R.T) / 1e6
Distances = squareform(pdist(rotated))

Omg = (Distances <= 0.22).astype(int)
np.fill_diagonal(Omg, 0)
Omg = csr_matrix(Omg)

deg = np.array(Omg.sum(axis=1)).flatten()
assert np.all(deg > 0), "Isolated points still exist!"

D = diags(deg)
Q = D - Omg

# ================================================================
# Scale ICAR (BYM2 requirement)
# ================================================================
def scale_icar_precision(Q):
    Q = Q.tocsc()
    eps = 1e-5
    factor = cholesky(Q + eps * diags(np.ones(Q.shape[0])))
    Vinv = factor.inv()
    marg_var = np.array(Vinv.diagonal()).flatten()
    scale = np.exp(np.mean(np.log(marg_var)))
    return Q / scale

Q_star = scale_icar_precision(Q)

# ================================================================
# MCMC settings
# ================================================================
burn = 1000
thin = 5
tot_save = 1000

a_tau = 0.001
b_tau = 0.001

phi_k_fixed = 0.7

# ================================================================
# BYM2 runner (FULL covariates)
# ================================================================
def run_bym2(event_name, loc_mask, kappa_transform, save_path):

    loc = np.where(loc_mask)
    pairs = np.column_stack(loc)
    pairs = pairs[np.lexsort((pairs[:, 0], pairs[:, 1]))]
    pairs[:, 1] += 1

    row_idx  = pairs[:, 0]
    time_idx = pairs[:, 1] - 1
    N = len(row_idx)

    next_y = y[row_idx, time_idx]
    kappa  = kappa_transform(next_y)

    # ------------------------------------------------------------
    # Time covariates
    # ------------------------------------------------------------
    t_raw   = time_idx + 1
    t_trend = t_trend_full[time_idx]

    X_cov = np.column_stack([
        np.ones(N),
        np.cos(2*np.pi*t_raw / period),
        np.sin(2*np.pi*t_raw / period),
        t_trend
    ])

    K = X_cov.shape[1]
    theta_dim = 2 * K * S

    # ------------------------------------------------------------
    # Design matrix
    # ------------------------------------------------------------
    rows, cols, vals = [], [], []

    for i in tqdm(range(N), desc=f"Design {event_name}"):
        s = row_idx[i]
        for k in range(K):
            x = X_cov[i, k]

            # u*_k
            rows.append(i)
            cols.append((2*k)*S + s)
            vals.append(x)

            # v_k
            rows.append(i)
            cols.append((2*k+1)*S + s)
            vals.append(x)

    X = coo_matrix((vals, (rows, cols)), shape=(N, theta_dim)).tocsr()

    # ------------------------------------------------------------
    # Storage
    # ------------------------------------------------------------
    total_iters = burn + tot_save * thin

    all_tau   = np.zeros((K, tot_save))
    all_theta = np.zeros((theta_dim, tot_save))

    curr_theta = np.zeros(theta_dim)
    curr_tau   = np.ones(K)

    save_idx = 0

    # ------------------------------------------------------------
    # MCMC
    # ------------------------------------------------------------
    for it in tqdm(range(total_iters), desc=f"MCMC {event_name}"):

        phi = X @ curr_theta
        omega = random_polyagamma(1, phi, size=N)

        # --------------------------------------------------------
        # Prior precision
        # --------------------------------------------------------
        blocks = []

        for k in range(K):
            tau = curr_tau[k]

            Qk = bmat([
                [tau * phi_k_fixed * Q_star, None],
                [None, tau * (1 - phi_k_fixed) * diags(np.ones(S))]
            ], format="csr")

            blocks.append(Qk)

        prior_prec = bmat(
            [[blocks[i] if i == j else None for j in range(K)] for i in range(K)],
            format="csr"
        )

        XtOmega = X.T.multiply(omega)
        post_prec = XtOmega @ X + prior_prec

        # ---- numerical stabilization ----
        post_prec = post_prec + 1e-6 * diags(np.ones(theta_dim))

        factor = cholesky(post_prec)
        mu = factor.solve_A(X.T @ kappa)
        curr_theta = mu + factor.solve_A(np.random.randn(theta_dim))

        # --------------------------------------------------------
        # τ_k updates
        # --------------------------------------------------------
        for k in range(K):
            u = curr_theta[(2*k)*S:(2*k+1)*S]
            v = curr_theta[(2*k+1)*S:(2*k+2)*S]

            quad = (
                phi_k_fixed * (u @ (Q_star @ u)) +
                (1 - phi_k_fixed) * (v @ v)
            )

            curr_tau[k] = np.random.gamma(
                a_tau + S,
                1 / (b_tau + 0.5 * quad)
            )

        if it >= burn and (it - burn) % thin == 0:
            all_tau[:, save_idx]   = curr_tau
            all_theta[:, save_idx] = curr_theta
            save_idx += 1
            if save_idx == tot_save:
                break

    np.savez_compressed(
        save_path,
        tau=all_tau,
        theta=all_theta,
        phi=phi_k_fixed
    )

# ================================================================
# Run p01
# ================================================================
run_bym2(
    event_name="p01",
    loc_mask=(y[:, :-1] == 0),
    kappa_transform=lambda ny: ny - 0.5,
    save_path=r"D:\77\Research\temp\snow\bym2_p01_phi07.npz"
)

# ================================================================
# Run p10
# ================================================================
run_bym2(
    event_name="p10",
    loc_mask=(y[:, :-1] == 1),
    kappa_transform=lambda ny: (1 - ny) - 0.5,
    save_path=r"D:\77\Research\temp\snow\bym2_p10_phi07.npz"
)


[INFO] Spatial locations used: 1601


MCMC p01:   0%|          | 0/6000 [00:00<?, ?it/s]C:\Users\lix23\AppData\Local\Temp\ipykernel_7572\3299437849.py:211: CholmodTypeConversionWarning: converting matrix of class csr_matrix to CSC format
  factor = cholesky(post_prec)
MCMC p10: 100%|█████████▉| 5995/6000 [2:38:50<00:07,  1.59s/it]  
